# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)


## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```


## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$


## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): +10.0,
            (2, 2): +2.0,
            (3, 5): -10.0,
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = 0.60
            p_dev_left = 0.20
            p_dev_right = 0.20
        else:
            p_intended = 0.90
            p_dev_left = 0.05
            p_dev_right = 0.05

        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_dev_left),
            (perp2, p_dev_right),
        ]

        trans_dict = {}
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )
            if not self.is_valid_state(next_state):
                next_state = state
            trans_dict[next_state] = trans_dict.get(next_state, 0.0) + prob

        return list(trans_dict.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    """Calcula sum_{s'} T(s,a,s') V(s')"""
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    """Algoritmo de Value Iteration para encontrar V*(s)"""
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = grid.get_reward(state) + grid.gamma * best_expected

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    """Extrae la política óptima greedy pi*(s) = argmax_a sum T(s,a,s') V*(s')"""
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$


In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    """Calcula V^pi(s) para una política fija pi"""
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    """Mejora la política buscando la acción greedy respecto a V^pi"""
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    """Algoritmo completo de Policy Iteration"""
    initial_action = (0, 1)  # RIGHT
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []

    for iteration in range(max_iter):
        V, eval_sweeps = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)

        changed = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )
        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_sweeps,
            "changed_actions": changed,
        })

        policy = new_policy
        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history


## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Respuestas:

1. **Preferencia de destino:** Desde `START` $(0, 0)$, el agente se dirige a la **estación de carga +2** en $(2, 2)$ siguiendo la ruta $(0,0) \to (0,1) \to (0,2) \to (1,2) \to (2,2)$. No busca la entrega $+10$.
2. **Optimalidad de recompensa menor:** Por la combinación del costo por paso $R(s) = -1.0$ y el descuento $\gamma = 0.9$. Llegar a la entrega requiere mínimo 7 pasos, acumulando costos negativos y un factor $\gamma^7 \approx 0.478$. El retorno esperado neto hacia $+10$ resulta inferior al de alcanzar la carga $+2$ en solo 4 pasos.
3. **Efecto del piso resbaloso:** En celdas como $(1, 2)$ y $(2, 1)$, donde la probabilidad de desviación lateral sube al $40\%$ ($20\%$ a cada lado). El agente adopta acciones más conservadoras para evitar caer accidentalmente en los peligros $(-3)$ o colisionar con estanterías.
4. **Papel del costo por paso:** Penaliza la demora temporal. Al ser fuertemente negativo ($-1.0$), incentiva al agente a terminar el episodio en el menor número de pasos posible, priorizando metas cercanas.
5. **Heterogeneidad de $T(s, a, s')$:** Porque las propiedades físicas de fricción dependen del estado actual $s$: el piso estándar tiene $90\%$ de probabilidad de éxito y $5\%-5\%$ lateral, mientras que el piso resbaloso tiene $60\%$ de éxito y $20\%-20\%$ lateral.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [ ]:
# ============================================================
# EXPERIMENTOS A, B, C y BONUS
# ============================================================

class WarehouseExperiment(WarehouseMDP):
    def __init__(self, living_reward=-1.0, gamma=0.9, slippery_prob=0.60):
        super().__init__()
        self.living_reward = living_reward
        self.gamma = gamma
        self.slippery_prob = slippery_prob

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]
        if state in self.slippery_states:
            p_intended = self.slippery_prob
            p_dev = (1.0 - p_intended) / 2.0
        else:
            p_intended = 0.90
            p_dev = 0.05
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])
        outcomes = [(action, p_intended), (perp1, p_dev), (perp2, p_dev)]
        trans_dict = {}
        for act, prob in outcomes:
            ns = (state[0] + act[0], state[1] + act[1])
            if not self.is_valid_state(ns):
                ns = state
            trans_dict[ns] = trans_dict.get(ns, 0.0) + prob
        return list(trans_dict.items())

# Experimento A: living_reward = -0.1
grid_A = WarehouseExperiment(living_reward=-0.1)
V_A, _ = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)
print("=== EXPERIMENTO A (living_reward = -0.1) ===")
print_policy(grid_A, pi_A)

# Experimento B: piso muy resbaloso (0.40 deseado, 0.30/0.30 desviaciones)
grid_B = WarehouseExperiment(slippery_prob=0.40)
V_B, _ = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)
print("\n=== EXPERIMENTO B (slippery_prob = 0.40) ===")
print_policy(grid_B, pi_B)

# Experimento C: gamma = 0.99
grid_C = WarehouseExperiment(gamma=0.99)
V_C, _ = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)
print("\n=== EXPERIMENTO C (gamma = 0.99) ===")
print_policy(grid_C, pi_C)

# Bonus: Umbral de living_reward donde la trayectoria desde START cambia hacia entrega +10
for lr in np.linspace(-1.0, 0.0, 101):
    g = WarehouseExperiment(living_reward=lr)
    V_tmp, _ = value_iteration(g)
    pi_tmp = extract_policy(g, V_tmp)
    curr = g.start
    for _ in range(15):
        if g.is_terminal(curr):
            break
        curr = (curr[0] + pi_tmp[curr][0], curr[1] + pi_tmp[curr][1])
    if curr == (0, 5):
        print(f"\n✓ BONUS: El umbral aproximado de living_reward es {lr:.3f}.")
        print(f"  Con penalizaciones mayores a {lr:.3f} (ej. -0.5 o -0.1), el robot prefiere la entrega +10.")
        break
